# 02 — Feature Engineering

This notebook creates model features using historical matches involving the 48 FIFA World Cup 2026 teams.

## Project Setup

The processed datasets from the preprocessing step are loaded.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path("..").resolve()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "output"
TABLES_DIR = OUTPUT_DIR / "tables"

TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Processed data folder:", PROCESSED_DIR)


Project folder: C:\Users\esmae\Documents\Educx Kurs machine lerning\aufgabe\ML_Abschlussprojekt_WorldCup2026
Processed data folder: C:\Users\esmae\Documents\Educx Kurs machine lerning\aufgabe\ML_Abschlussprojekt_WorldCup2026\data\processed


## Load Processed Data

The cleaned historical match data, World Cup 2026 teams, and fixture data are imported.

In [2]:
clean_matches = pd.read_csv(PROCESSED_DIR / "clean_all_matches.csv")
teams_2026 = pd.read_csv(PROCESSED_DIR / "wc2026_teams.csv")
fixtures = pd.read_csv(PROCESSED_DIR / "wc2026_fixtures.csv")
h2h_summary = pd.read_csv(PROCESSED_DIR / "wc2026_fixture_h2h_summary.csv")

clean_matches["date"] = pd.to_datetime(clean_matches["date"], errors="coerce")
fixtures["date"] = pd.to_datetime(fixtures["date"], errors="coerce")
h2h_summary["date"] = pd.to_datetime(h2h_summary["date"], errors="coerce")

print("Clean matches:", clean_matches.shape)
print("World Cup 2026 teams:", teams_2026.shape)
print("Fixtures:", fixtures.shape)
print("H2H summary:", h2h_summary.shape)

display(clean_matches.head())
display(teams_2026.head())
display(fixtures.head())


Clean matches: (51644, 12)
World Cup 2026 teams: (112, 3)
Fixtures: (104, 12)
H2H summary: (104, 16)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_team_clean,away_team_clean,result
0,1872-11-30,Scotland,England,0,0,Friendly,NaN,Scotland,False,Scotland,England,draw
1,1873-03-08,England,Scotland,4,2,Friendly,NaN,England,False,England,Scotland,home_win
2,1874-03-07,Scotland,England,2,1,Friendly,NaN,Scotland,False,Scotland,England,home_win
3,1875-03-06,England,Scotland,2,2,Friendly,NaN,England,False,England,Scotland,draw
4,1876-03-04,Scotland,England,3,0,Friendly,NaN,Scotland,False,Scotland,England,home_win


,team,team_clean,group
0,Czechia/Denmark/North Macedonia/Republic of Ir...,Czechia/Denmark/North Macedonia/Republic of Ir...,Group A
1,Korea Republic,Korea Republic,Group A
2,Mexico,Mexico,Group A
3,South Africa,South Africa,Group A
4,Bosnia and Herzegovina/Italy/Northern Ireland/...,Bosnia and Herzegovina/Italy/Northern Ireland/...,Group B


,match_no,date,stage,group,team_a,team_b,team_a_clean,team_b_clean,venue,city,country,neutral
0,Match 1,2026-06-11,NaN,Group A,Mexico,South Africa,Mexico,South Africa,Mexico City Stadium,NaN,NaN,True
1,Match 2,2026-06-11,NaN,Group A,Korea Republic,Czechia/Denmark/North Macedonia/Republic of Ir...,Korea Republic,Czechia/Denmark/North Macedonia/Republic of Ir...,Estadio Guadalajara,NaN,NaN,True
2,Match 3,2026-06-12,NaN,Group B,Canada,Bosnia and Herzegovina/Italy/Northern Ireland/...,Canada,Bosnia and Herzegovina/Italy/Northern Ireland/...,Toronto Stadium,NaN,NaN,True
3,Match 4,2026-06-12,NaN,Group D,USA,Paraguay,USA,Paraguay,Los Angeles Stadium,NaN,NaN,True
4,Match 5,2026-06-13,NaN,Group C,Haiti,Scotland,Haiti,Scotland,Boston Stadium,NaN,NaN,True


## Base Dataset Filter

The historical dataset is filtered to matches involving at least one of the 48 World Cup 2026 teams.

In [3]:
teams_list = teams_2026["team_clean"].dropna().unique().tolist()

base_matches = clean_matches[
    clean_matches["home_team_clean"].isin(teams_list) |
    clean_matches["away_team_clean"].isin(teams_list)
].dropna(
    subset=["date", "home_team_clean", "away_team_clean", "home_score", "away_score"]
).copy()

base_matches = base_matches[base_matches["date"] >= "1995-01-01"].copy()
base_matches = base_matches.sort_values("date").reset_index(drop=True)

print("Original historical matches:", clean_matches.shape)
print("Filtered matches involving 2026 teams from 1995:", base_matches.shape)

display(base_matches.head())
display(base_matches.tail())

base_matches.to_csv(PROCESSED_DIR / "base_matches_2026_teams_from_1995.csv", index=False)
print("Saved:", PROCESSED_DIR / "base_matches_2026_teams_from_1995.csv")


Original historical matches: (51644, 12)
Filtered matches involving 2026 teams from 1995: (12900, 12)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_team_clean,away_team_clean,result
0,1995-01-02,Tunisia,Egypt,2,0,Friendly,NaN,Tunisia,False,Tunisia,Egypt,home_win
1,1995-01-06,Nigeria,Japan,3,0,Intercontinental Champ,NaN,Saudi Arabia,True,Nigeria,Japan,home_win
2,1995-01-06,Saudi Arabia,Mexico,0,2,Intercontinental Champ,NaN,Saudi Arabia,False,Saudi Arabia,Mexico,away_win
3,1995-01-07,Senegal,Tunisia,0,0,African Nations Cup qualifier,NaN,Senegal,False,Senegal,Tunisia,draw
4,1995-01-08,Algeria,Egypt,1,0,African Nations Cup qualifier,NaN,Algeria,False,Algeria,Egypt,home_win


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_team_clean,away_team_clean,result
12895,2026-06-08,Spain,Peru,3,1,Friendly,NaN,Mexico,True,Spain,Peru,home_win
12896,2026-06-09,Saudi Arabia,Senegal,0,0,Friendly,NaN,United States,True,Saudi Arabia,Senegal,draw
12897,2026-06-09,Argentina,Iceland,3,0,Friendly,NaN,United States,True,Argentina,Iceland,home_win
12898,2026-06-10,Portugal,Nigeria,2,1,Friendly,NaN,Portugal,False,Portugal,Nigeria,home_win
12899,2026-06-10,England,Costa Rica,3,0,Friendly,NaN,United States,True,England,Costa Rica,home_win


Saved: C:\Users\esmae\Documents\Educx Kurs machine lerning\aufgabe\ML_Abschlussprojekt_WorldCup2026\data\processed\base_matches_2026_teams_from_1995.csv


## Training Size Control

The most recent matches are selected to keep feature engineering computationally efficient.

In [4]:
MAX_TRAINING_ROWS = 8000

if len(base_matches) > MAX_TRAINING_ROWS:
    historical_matches = base_matches.tail(MAX_TRAINING_ROWS).copy()
else:
    historical_matches = base_matches.copy()

historical_matches = historical_matches.sort_values("date").reset_index(drop=True)

train_rows_80 = int(len(historical_matches) * 0.8)
test_rows_20 = len(historical_matches) - train_rows_80

print("Training feature rows used:", len(historical_matches))
print("Estimated train rows 80%:", train_rows_80)
print("Estimated test rows 20%:", test_rows_20)

display(historical_matches.head())
display(historical_matches.tail())


Training feature rows used: 8000
Estimated train rows 80%: 6400
Estimated test rows 20%: 1600


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_team_clean,away_team_clean,result
0,2007-03-24,Algeria,Cape Verde,2,0,African Nations Cup qualifier,NaN,Algeria,False,Algeria,Cabo Verde,home_win
1,2007-03-24,Portugal,Belgium,4,0,European Championship qual,NaN,Portugal,False,Portugal,Belgium,home_win
2,2007-03-24,Norway,Bosnia and Herzegovina,1,2,European Championship qual,NaN,Norway,False,Norway,Bosnia and Herzegovina,away_win
3,2007-03-24,Netherlands,Romania,0,0,European Championship qual,NaN,Netherlands,False,Netherlands,Romania,draw
4,2007-03-24,Czechia,Germany,1,2,European Championship qual,NaN,Czechia,False,Czechia,Germany,away_win


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_team_clean,away_team_clean,result
7995,2026-06-08,Spain,Peru,3,1,Friendly,NaN,Mexico,True,Spain,Peru,home_win
7996,2026-06-09,Saudi Arabia,Senegal,0,0,Friendly,NaN,United States,True,Saudi Arabia,Senegal,draw
7997,2026-06-09,Argentina,Iceland,3,0,Friendly,NaN,United States,True,Argentina,Iceland,home_win
7998,2026-06-10,Portugal,Nigeria,2,1,Friendly,NaN,Portugal,False,Portugal,Nigeria,home_win
7999,2026-06-10,England,Costa Rica,3,0,Friendly,NaN,United States,True,England,Costa Rica,home_win


## Feature Functions

Recent team performance and direct head-to-head statistics are calculated before each match date.

In [5]:
def get_team_history(df, team, match_date):
    history = df[
        ((df["home_team_clean"] == team) | (df["away_team_clean"] == team)) &
        (df["date"] < match_date)
    ].sort_values("date")

    return history


def get_recent_team_stats(df, team, match_date, n_matches=10):
    history = get_team_history(df, team, match_date).tail(n_matches)

    if history.empty:
        return {
            "matches": 0,
            "win_rate": 0.0,
            "draw_rate": 0.0,
            "loss_rate": 0.0,
            "avg_goals_scored": 0.0,
            "avg_goals_conceded": 0.0,
            "avg_goal_diff": 0.0,
            "points_per_match": 0.0,
        }

    wins = 0
    draws = 0
    losses = 0
    goals_scored = []
    goals_conceded = []
    goal_diffs = []
    points = []

    for _, row in history.iterrows():
        if row["home_team_clean"] == team:
            scored = row["home_score"]
            conceded = row["away_score"]
        else:
            scored = row["away_score"]
            conceded = row["home_score"]

        if pd.isna(scored) or pd.isna(conceded):
            continue

        goals_scored.append(scored)
        goals_conceded.append(conceded)
        goal_diffs.append(scored - conceded)

        if scored > conceded:
            wins += 1
            points.append(3)
        elif scored < conceded:
            losses += 1
            points.append(0)
        else:
            draws += 1
            points.append(1)

    valid_matches = len(points)

    if valid_matches == 0:
        return {
            "matches": 0,
            "win_rate": 0.0,
            "draw_rate": 0.0,
            "loss_rate": 0.0,
            "avg_goals_scored": 0.0,
            "avg_goals_conceded": 0.0,
            "avg_goal_diff": 0.0,
            "points_per_match": 0.0,
        }

    return {
        "matches": valid_matches,
        "win_rate": wins / valid_matches,
        "draw_rate": draws / valid_matches,
        "loss_rate": losses / valid_matches,
        "avg_goals_scored": np.mean(goals_scored),
        "avg_goals_conceded": np.mean(goals_conceded),
        "avg_goal_diff": np.mean(goal_diffs),
        "points_per_match": np.mean(points),
    }


def get_pair_h2h_stats(df, team_a, team_b, match_date):
    h2h = df[
        (
            (df["home_team_clean"] == team_a) &
            (df["away_team_clean"] == team_b)
        ) |
        (
            (df["home_team_clean"] == team_b) &
            (df["away_team_clean"] == team_a)
        )
    ]

    h2h = h2h[h2h["date"] < match_date].sort_values("date")

    if h2h.empty:
        return {
            "h2h_matches": 0,
            "team_a_h2h_win_rate": 0.0,
            "team_b_h2h_win_rate": 0.0,
            "h2h_draw_rate": 0.0,
            "team_a_h2h_avg_goals": 0.0,
            "team_b_h2h_avg_goals": 0.0,
        }

    team_a_wins = 0
    team_b_wins = 0
    draws = 0
    goals_a = []
    goals_b = []

    for _, row in h2h.iterrows():
        if row["home_team_clean"] == team_a:
            a_score = row["home_score"]
            b_score = row["away_score"]
        else:
            a_score = row["away_score"]
            b_score = row["home_score"]

        if pd.isna(a_score) or pd.isna(b_score):
            continue

        goals_a.append(a_score)
        goals_b.append(b_score)

        if a_score > b_score:
            team_a_wins += 1
        elif a_score < b_score:
            team_b_wins += 1
        else:
            draws += 1

    valid_matches = len(goals_a)

    if valid_matches == 0:
        return {
            "h2h_matches": 0,
            "team_a_h2h_win_rate": 0.0,
            "team_b_h2h_win_rate": 0.0,
            "h2h_draw_rate": 0.0,
            "team_a_h2h_avg_goals": 0.0,
            "team_b_h2h_avg_goals": 0.0,
        }

    return {
        "h2h_matches": valid_matches,
        "team_a_h2h_win_rate": team_a_wins / valid_matches,
        "team_b_h2h_win_rate": team_b_wins / valid_matches,
        "h2h_draw_rate": draws / valid_matches,
        "team_a_h2h_avg_goals": np.mean(goals_a),
        "team_b_h2h_avg_goals": np.mean(goals_b),
    }


## Match Feature Builder

Each match is represented by recent form, goal performance, differences between teams, and head-to-head features.

In [6]:
def create_match_features(df, team_a, team_b, match_date, neutral=True, tournament=None):
    a_last5 = get_recent_team_stats(df, team_a, match_date, n_matches=5)
    b_last5 = get_recent_team_stats(df, team_b, match_date, n_matches=5)

    a_last10 = get_recent_team_stats(df, team_a, match_date, n_matches=10)
    b_last10 = get_recent_team_stats(df, team_b, match_date, n_matches=10)

    h2h = get_pair_h2h_stats(df, team_a, team_b, match_date)

    features = {
        "team_a": team_a,
        "team_b": team_b,
        "date": match_date,
        "tournament": tournament,
        "neutral": neutral,

        "team_a_matches_last5": a_last5["matches"],
        "team_b_matches_last5": b_last5["matches"],
        "team_a_win_rate_last5": a_last5["win_rate"],
        "team_b_win_rate_last5": b_last5["win_rate"],
        "team_a_draw_rate_last5": a_last5["draw_rate"],
        "team_b_draw_rate_last5": b_last5["draw_rate"],
        "team_a_loss_rate_last5": a_last5["loss_rate"],
        "team_b_loss_rate_last5": b_last5["loss_rate"],
        "team_a_points_per_match_last5": a_last5["points_per_match"],
        "team_b_points_per_match_last5": b_last5["points_per_match"],

        "team_a_matches_last10": a_last10["matches"],
        "team_b_matches_last10": b_last10["matches"],
        "team_a_win_rate_last10": a_last10["win_rate"],
        "team_b_win_rate_last10": b_last10["win_rate"],
        "team_a_draw_rate_last10": a_last10["draw_rate"],
        "team_b_draw_rate_last10": b_last10["draw_rate"],
        "team_a_loss_rate_last10": a_last10["loss_rate"],
        "team_b_loss_rate_last10": b_last10["loss_rate"],
        "team_a_avg_goals_scored_last10": a_last10["avg_goals_scored"],
        "team_b_avg_goals_scored_last10": b_last10["avg_goals_scored"],
        "team_a_avg_goals_conceded_last10": a_last10["avg_goals_conceded"],
        "team_b_avg_goals_conceded_last10": b_last10["avg_goals_conceded"],
        "team_a_avg_goal_diff_last10": a_last10["avg_goal_diff"],
        "team_b_avg_goal_diff_last10": b_last10["avg_goal_diff"],
        "team_a_points_per_match_last10": a_last10["points_per_match"],
        "team_b_points_per_match_last10": b_last10["points_per_match"],

        "win_rate_diff_last10": a_last10["win_rate"] - b_last10["win_rate"],
        "points_per_match_diff_last10": a_last10["points_per_match"] - b_last10["points_per_match"],
        "avg_goals_scored_diff_last10": a_last10["avg_goals_scored"] - b_last10["avg_goals_scored"],
        "avg_goals_conceded_diff_last10": a_last10["avg_goals_conceded"] - b_last10["avg_goals_conceded"],
        "avg_goal_diff_diff_last10": a_last10["avg_goal_diff"] - b_last10["avg_goal_diff"],

        **h2h,
    }

    return features


## Historical Training Rows

The filtered historical matches are transformed into a supervised learning dataset.

In [7]:
rows = []

for idx, row in historical_matches.iterrows():
    match_date = row["date"]
    team_a = row["home_team_clean"]
    team_b = row["away_team_clean"]

    features = create_match_features(
        base_matches,
        team_a=team_a,
        team_b=team_b,
        match_date=match_date,
        neutral=row.get("neutral", True),
        tournament=row.get("tournament", "Unknown"),
    )

    features["team_a_goals"] = row["home_score"]
    features["team_b_goals"] = row["away_score"]

    if row["home_score"] > row["away_score"]:
        features["result"] = "team_a_win"
    elif row["home_score"] < row["away_score"]:
        features["result"] = "team_b_win"
    else:
        features["result"] = "draw"

    rows.append(features)

model_dataset = pd.DataFrame(rows)

print("Model dataset shape:", model_dataset.shape)
display(model_dataset.head())


Model dataset shape: (8000, 45)


,team_a,team_b,date,tournament,neutral,team_a_matches_last5,team_b_matches_last5,team_a_win_rate_last5,team_b_win_rate_last5,team_a_draw_rate_last5,...,avg_goal_diff_diff_last10,h2h_matches,team_a_h2h_win_rate,team_b_h2h_win_rate,h2h_draw_rate,team_a_h2h_avg_goals,team_b_h2h_avg_goals,team_a_goals,team_b_goals,result
0,Algeria,Cabo Verde,2007-03-24,African Nations Cup qualifier,False,5,5,0.4,0.2,0.2,...,0.7,2,0.50,0.00,0.5,1.00,0.000000,2,0,team_a_win
1,Portugal,Belgium,2007-03-24,European Championship qual,False,5,5,0.6,0.4,0.2,...,0.0,1,0.00,0.00,1.0,1.00,1.000000,4,0,team_a_win
2,Norway,Bosnia and Herzegovina,2007-03-24,European Championship qual,False,5,5,0.4,0.2,0.2,...,0.6,2,0.50,0.50,0.0,1.00,0.500000,1,2,team_b_win
3,Netherlands,Romania,2007-03-24,European Championship qual,False,5,5,0.6,0.2,0.4,...,1.3,3,1.00,0.00,0.0,2.00,0.333333,0,0,draw
4,Czechia,Germany,2007-03-24,European Championship qual,False,5,5,0.8,0.8,0.0,...,-1.9,4,0.25,0.75,0.0,1.25,2.000000,1,2,team_b_win


## Dataset Quality Check

The feature dataset is checked for missing values and target distribution.

In [8]:
print("Missing values by column:")
display(model_dataset.isna().sum().sort_values(ascending=False).head(20))

print("Result distribution:")
display(model_dataset["result"].value_counts(normalize=True).rename("share").to_frame())

print("Goal summary:")
display(model_dataset[["team_a_goals", "team_b_goals"]].describe())


Missing values by column:


team_a                           0
team_b                           0
date                             0
tournament                       0
neutral                          0
team_a_matches_last5             0
team_b_matches_last5             0
team_a_win_rate_last5            0
team_b_win_rate_last5            0
team_a_draw_rate_last5           0
team_b_draw_rate_last5           0
team_a_loss_rate_last5           0
team_b_loss_rate_last5           0
team_a_points_per_match_last5    0
team_b_points_per_match_last5    0
team_a_matches_last10            0
team_b_matches_last10            0
team_a_win_rate_last10           0
team_b_win_rate_last10           0
team_a_draw_rate_last10          0
dtype: int64

Result distribution:


,share
result,
team_a_win,0.576125
draw,0.233625
team_b_win,0.190250


Goal summary:


,team_a_goals,team_b_goals
count,8000.000000,8000.000000
mean,1.775500,0.892875
std,1.584744,1.117388
min,0.000000,0.000000
25%,1.000000,0.000000
50%,1.000000,1.000000
75%,3.000000,1.000000
max,15.000000,10.000000


## World Cup 2026 Fixture Features

The same feature logic is applied to upcoming World Cup 2026 fixtures.

In [9]:
fixture_rows = []

for _, row in fixtures.iterrows():
    match_date = row["date"]
    team_a = row["team_a_clean"]
    team_b = row["team_b_clean"]

    features = create_match_features(
        base_matches,
        team_a=team_a,
        team_b=team_b,
        match_date=match_date,
        neutral=row.get("neutral", True),
        tournament="FIFA World Cup 2026",
    )

    features["match_no"] = row.get("match_no", np.nan)
    features["stage"] = row.get("stage", np.nan)
    features["group"] = row.get("group", np.nan)
    features["venue"] = row.get("venue", np.nan)
    features["city"] = row.get("city", np.nan)
    features["country"] = row.get("country", np.nan)

    fixture_rows.append(features)

fixture_feature_dataset = pd.DataFrame(fixture_rows)

print("Fixture feature dataset shape:", fixture_feature_dataset.shape)
display(fixture_feature_dataset.head())


Fixture feature dataset shape: (104, 48)


,team_a,team_b,date,tournament,neutral,team_a_matches_last5,team_b_matches_last5,team_a_win_rate_last5,team_b_win_rate_last5,team_a_draw_rate_last5,...,team_b_h2h_win_rate,h2h_draw_rate,team_a_h2h_avg_goals,team_b_h2h_avg_goals,match_no,stage,group,venue,city,country
0,Mexico,South Africa,2026-06-11,FIFA World Cup 2026,True,5,5,0.6,0.2,0.4,...,0.333333,0.333333,2.000,1.666667,Match 1,NaN,Group A,Mexico City Stadium,NaN,NaN
1,Korea Republic,Czechia/Denmark/North Macedonia/Republic of Ir...,2026-06-11,FIFA World Cup 2026,True,5,0,0.6,0.0,0.0,...,0.000000,0.000000,0.000,0.000000,Match 2,NaN,Group A,Estadio Guadalajara,NaN,NaN
2,Canada,Bosnia and Herzegovina/Italy/Northern Ireland/...,2026-06-12,FIFA World Cup 2026,True,5,0,0.4,0.0,0.6,...,0.000000,0.000000,0.000,0.000000,Match 3,NaN,Group B,Toronto Stadium,NaN,NaN
3,USA,Paraguay,2026-06-12,FIFA World Cup 2026,True,5,5,0.4,0.6,0.0,...,0.250000,0.250000,1.125,0.875000,Match 4,NaN,Group D,Los Angeles Stadium,NaN,NaN
4,Haiti,Scotland,2026-06-13,FIFA World Cup 2026,True,5,5,0.4,0.6,0.2,...,0.000000,0.000000,0.000,0.000000,Match 5,NaN,Group C,Boston Stadium,NaN,NaN


## Export Feature Datasets

The engineered datasets are saved for model training and tournament prediction.

In [10]:
model_dataset.to_csv(PROCESSED_DIR / "model_dataset.csv", index=False)
fixture_feature_dataset.to_csv(PROCESSED_DIR / "wc2026_fixture_features.csv", index=False)

model_dataset.head(20).to_csv(TABLES_DIR / "model_dataset_preview.csv", index=False)
fixture_feature_dataset.head(20).to_csv(TABLES_DIR / "fixture_feature_preview.csv", index=False)

print("Saved:", PROCESSED_DIR / "model_dataset.csv")
print("Saved:", PROCESSED_DIR / "wc2026_fixture_features.csv")
print("Saved:", TABLES_DIR / "model_dataset_preview.csv")
print("Saved:", TABLES_DIR / "fixture_feature_preview.csv")


Saved: C:\Users\esmae\Documents\Educx Kurs machine lerning\aufgabe\ML_Abschlussprojekt_WorldCup2026\data\processed\model_dataset.csv
Saved: C:\Users\esmae\Documents\Educx Kurs machine lerning\aufgabe\ML_Abschlussprojekt_WorldCup2026\data\processed\wc2026_fixture_features.csv
Saved: C:\Users\esmae\Documents\Educx Kurs machine lerning\aufgabe\ML_Abschlussprojekt_WorldCup2026\output\tables\model_dataset_preview.csv
Saved: C:\Users\esmae\Documents\Educx Kurs machine lerning\aufgabe\ML_Abschlussprojekt_WorldCup2026\output\tables\fixture_feature_preview.csv


## Feature Engineering Summary

The project now contains a filtered training dataset and a World Cup 2026 fixture feature dataset.

In [11]:
print("Final outputs:")
print(" - data/processed/base_matches_2026_teams_from_1995.csv")
print(" - data/processed/model_dataset.csv")
print(" - data/processed/wc2026_fixture_features.csv")
print(" - output/tables/model_dataset_preview.csv")
print(" - output/tables/fixture_feature_preview.csv")

print("Feature engineering completed.")


Final outputs:
 - data/processed/base_matches_2026_teams_from_1995.csv
 - data/processed/model_dataset.csv
 - data/processed/wc2026_fixture_features.csv
 - output/tables/model_dataset_preview.csv
 - output/tables/fixture_feature_preview.csv
Feature engineering completed.
